# Unsupervised Learning: Hands-On Companion to the Slides

This notebook follows the slides on unsupervised learning, one block per slide. Each block redraws what the slide shows; then you change one setting and run the cell again.

**How to use it**

1. Run the setup cell once.
2. When a slide comes up, find the block with the same title (Colab's table of contents in the left sidebar lists them) and run its code cell.
3. The settings sit at the top of each code cell in capitals, e.g. `K = 3`. Change one, rerun the cell and compare with the slide. The **Try** line under each block suggests what to change and what to look for.

Every block needs only the setup cell, except block 3, which reuses the functions of block 2. All data are simulated, as on the slides, except the US arrests data in the last block. The random numbers differ from those behind the slides, so your figures look like the slides without being identical. The whole notebook runs in well under a minute.

## Setup

In [ ]:
# Setup: libraries, the slides' colours, the three-blob data and a plotting helper
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import make_moons
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_samples, silhouette_score
from sklearn.preprocessing import StandardScaler

RAW = 'https://raw.githubusercontent.com/umatter/EDFB/main/'

NAVY, BLUE, GOLD, GREY, TEAL, ROSE = '#1F2A44', '#1F4E9C', '#E9BE2E', '#8A93A3', '#2A9D8F', '#C0567A'
COLS = [BLUE, GOLD, TEAL, ROSE, '#7B5EA7', '#E07A1F', '#5C946E', '#9C2F2F', '#3FA7D6', '#B07AA1']
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.titleweight': 'bold', 'axes.titlecolor': NAVY, 'axes.titlesize': 11})


def make_blobs_slides(sd=0.9, seed=409):
    '''The slides' three-blob data: 50 points around each of (2, 2.2), (8, 3) and (4.6, 8).'''
    rng = np.random.default_rng(seed)
    centres = np.array([[2.0, 2.2], [8.0, 3.0], [4.6, 8.0]])
    return np.vstack([rng.normal(c, sd, size=(50, 2)) for c in centres])


def show(X, labels=None, centers=None, ax=None, title=''):
    '''Scatter plot coloured by cluster (grey without labels, and for DBSCAN noise, label -1).
    Centroids are drawn as navy crosses.'''
    ax = ax if ax is not None else plt.gca()
    colours = GREY if labels is None else [GREY if l < 0 else COLS[l % len(COLS)] for l in labels]
    ax.scatter(X[:, 0], X[:, 1], s=14, c=colours)
    if centers is not None:
        ax.scatter(centers[:, 0], centers[:, 1], marker='x', s=110, linewidths=3, c=NAVY)
    ax.set_title(title)
    return ax


def agreement(truth, labels):
    '''Adjusted Rand index of two labellings, rounded: 1 = the same groups, 0 = no better than chance.'''
    return round(adjusted_rand_score(truth, labels), 2) + 0   # + 0 turns -0.0 into 0.0


X_blobs = make_blobs_slides()
print('Three-blob data:', X_blobs.shape[0], 'points,', X_blobs.shape[1], 'variables')

## 1. Before and after K-means

*Slide: What is cluster analysis?*

K-means takes points without labels (left) and splits them into `K` groups (right). The crosses are the centroids, the mean of each group.

In [ ]:
# K-means on the three blobs
K = 3   # number of clusters

km = KMeans(n_clusters=K, n_init=25, random_state=0).fit(X_blobs)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
show(X_blobs, ax=axes[0], title='Before K-means')
show(X_blobs, km.labels_, km.cluster_centers_, ax=axes[1], title=f'After K-means (k = {K})')
plt.tight_layout()
plt.show()

**Try:** set `K = 6`, then `K = 2`. K-means always returns exactly `K` clusters, whether or not the data have them. Which of the six clusters would you call a real group?

## 2. The K-means steps

*Slide: K-means (six panels)*

K-means repeats two steps until no point changes cluster:

1. **Assignment:** each point joins its nearest centroid.
2. **Update:** each centroid moves to the mean of its points.

The two functions `assign` and `update` below are the whole algorithm. The figure shows the start, the first two passes and the result.

In [ ]:
# The two steps of k-means, written out
def assign(X, centers):
    '''Step 1: the index of the nearest centroid for every point (squared Euclidean distance).'''
    d = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    return d.argmin(axis=1)


def update(X, labels, centers):
    '''Step 2: each centroid moves to the mean of its points (an empty cluster keeps its centroid).'''
    return np.array([X[labels == j].mean(axis=0) if (labels == j).any() else centers[j]
                     for j in range(len(centers))])


def kmeans_by_hand(X, centers, max_iter=50):
    '''Repeat step 1 and step 2 until no point changes cluster. Returns one (labels, centroids) pair per pass.'''
    history, labels = [], None
    for _ in range(max_iter):
        new_labels = assign(X, centers)
        if labels is not None and (new_labels == labels).all():
            break
        labels = new_labels
        centers = update(X, labels, centers)
        history.append((labels, centers))
    return history


K = 3        # number of clusters
SEED = 431   # seed for the random starting centroids

rng = np.random.default_rng(SEED)
start = X_blobs[rng.choice(len(X_blobs), K, replace=False)]   # K random data points as starting centroids
history = kmeans_by_hand(X_blobs, start)
(l1, c1), (l2, _), (l_end, c_end) = history[0], history[min(1, len(history) - 1)], history[-1]

panels = [(None, None, 'Data'),
          (None, start, 'Start: random centroids'),
          (l1, start, 'Pass 1, step 1: assign'),
          (l1, c1, 'Pass 1, step 2: update'),
          (l2, c1, 'Pass 2, step 1: assign'),
          (l_end, c_end, f'Result after {len(history)} passes')]
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, (labels, centers, title) in zip(axes.flat, panels):
    show(X_blobs, labels, centers, ax=ax, title=title)
plt.tight_layout()
plt.show()

**Try:** change `SEED` to start from other random points, or set `K = 4`. How many passes does it take now, and do you always end with the same result? Block 4 comes back to this.

## 3. The loop by hand

*Slide: K-means, "Its basic operation is very simple"*

The fifteen points and the three starting centroids from the slide. The code runs the same two steps as block 2 and prints each pass, so you can check the slide's arithmetic. Point X at (8, 3), ringed, starts blue and switches to teal on the second pass.

In [ ]:
# The slide's toy example: groups A, B and C, and point X
toy = np.array([[3, 9], [4.5, 10], [2.5, 7.5], [4, 8], [5, 9.5],         # group A
                [13, 10], [14.5, 9], [15, 10.5], [13.5, 8.2], [16, 9],    # group B
                [9.5, 2], [10, 3.5], [8.5, 1.5], [11, 2.5],               # group C
                [8, 3]])                                                  # point X
START = np.array([[6, 6], [10, 8.5], [12, 4]])   # starting centroids: blue, gold, teal

colour_names = ['blue', 'gold', 'teal', 'rose', 'purple', 'orange']
history = kmeans_by_hand(toy, START.astype(float))
for p, (labels, centers) in enumerate(history, start=1):
    print(f'Pass {p}: X is {colour_names[labels[-1]]}; centroids move to {centers.round(2).tolist()}')
print(f'Pass {len(history) + 1}: no point changes cluster, stop.')
wcss = sum(((toy[labels == j] - c) ** 2).sum() for j, c in enumerate(centers))
print(f'WCSS of the result: {wcss:.1f}')

fig, axes = plt.subplots(1, len(history) + 1, figsize=(3.8 * (len(history) + 1), 3.3))
show(toy, None, START, ax=axes[0], title='Start')
previous = START
for ax, (p, (labels, centers)) in zip(axes[1:], enumerate(history, start=1)):
    show(toy, labels, centers, ax=ax, title=f'Pass {p}')
    for a, b in zip(previous, centers):                       # how far each centroid moved
        ax.annotate('', xy=b, xytext=a, arrowprops=dict(arrowstyle='->', color=NAVY))
    previous = centers
for ax in axes:
    ax.scatter(*toy[-1], s=170, facecolors='none', edgecolors=NAVY)   # ring point X
plt.tight_layout()
plt.show()

**Try:** start from `START = np.array([[2, 8], [5, 10], [12, 6]])`. Two centroids now start inside group A: k-means splits A in two, lumps B and C together, and stops there, with a WCSS of about 199 instead of 26. The result depends on where you start, and that is the next slide.

## 4. Different starting values

*Slide: Starting different values*

**WCSS** (within-cluster sum of squares) adds up each point's squared distance to the centroid of its cluster; lower means tighter clusters. Below are six runs on the same data with `k = 3`, each from one random start. The six seeds were picked to show the problem: three of these runs get stuck in a worse solution. On these data most random starts do fine; 5 of the first 80 seeds get stuck.

In [ ]:
# Six single random starts, each titled by its WCSS
SEEDS = [0, 17, 1, 24, 2, 53]   # one random start per panel
N_INIT = 1                      # starts per run; KMeans keeps the one with the lowest WCSS

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, seed in zip(axes.flat, SEEDS):
    km = KMeans(n_clusters=3, init='random', n_init=N_INIT, random_state=seed).fit(X_blobs)
    show(X_blobs, km.labels_, km.cluster_centers_, ax=ax, title=f'seed {seed}: WCSS = {km.inertia_:.1f}')
plt.tight_layout()
plt.show()

**Try:** set `N_INIT = 10`. Each run now keeps the best of ten starts, and all six panels reach the same lowest WCSS. That is why every other block uses `n_init=25`. (scikit-learn calls the WCSS `inertia_`.)

## 5. Measuring fit: WCSS and BCSS

*Slides: Performance, and the within- and between-cluster variation figure*

- **WCSS**: how close together are the points within the clusters? The sum of squared distances of the points to their own centroid. Small is good.
- **BCSS**: how far apart are the clusters? The size-weighted sum of squared distances between each centroid and the overall mean. Large is good.

The code computes both from these definitions and checks that WCSS + BCSS equals the total sum of squares (TSS).

In [ ]:
# WCSS, BCSS and the total sum of squares
K = 3

km = KMeans(n_clusters=K, n_init=25, random_state=0).fit(X_blobs)
labels, centers = km.labels_, km.cluster_centers_
mean = X_blobs.mean(axis=0)
wcss = sum(((X_blobs[labels == j] - c) ** 2).sum() for j, c in enumerate(centers))
bcss = sum((labels == j).sum() * ((c - mean) ** 2).sum() for j, c in enumerate(centers))
tss = ((X_blobs - mean) ** 2).sum()
print(f'WCSS        = {wcss:7.1f}   (scikit-learn: km.inertia_ = {km.inertia_:.1f})')
print(f'BCSS        = {bcss:7.1f}')
print(f'WCSS + BCSS = {wcss + bcss:7.1f}')
print(f'TSS         = {tss:7.1f}')

plt.figure(figsize=(6.5, 5))
ax = show(X_blobs, labels, centers, title=f'Within (dashed, one cluster) and between (rose), k = {K}')
for x in X_blobs[labels == 0]:
    ax.plot(*zip(x, centers[0]), ls='--', lw=0.6, c=NAVY)   # within: the points of one cluster to their centroid
for c in centers:
    ax.plot(*zip(c, mean), lw=2, c=ROSE)                    # between: each centroid to the overall mean
ax.scatter(*mean, s=90, c=ROSE, zorder=3)
ax.annotate('overall mean', mean, xytext=(10, 6), textcoords='offset points', color=ROSE)
plt.show()

**Try:** set `K = 2`, `4` and `6`. WCSS falls and BCSS rises as `K` grows, while their sum stays the same, because the data fix the total. Making clusters tighter is the same as pushing them apart, so neither number can choose `K` by itself.

## 6. Choosing k: the elbow method

*Slide: Performance: how to choose optimal number of clusters? (elbow)*

WCSS for k = 1 to 10. It always falls as k grows; the elbow is where one more cluster stops buying much. It is a heuristic, not a proof.

In [ ]:
# Elbow curve: WCSS for k = 1..10
SD = 0.9   # spread of each blob (the slides use 0.9)

X = make_blobs_slides(sd=SD)
ks = range(1, 11)
wcss_k = [KMeans(n_clusters=k, n_init=25, random_state=0).fit(X).inertia_ for k in ks]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
show(X, ax=axes[0], title=f'Data (blob spread {SD})')
axes[1].plot(ks, wcss_k, 'o-', c=NAVY)
axes[1].scatter(3, wcss_k[2], s=130, c=GOLD, zorder=3, label='k = 3, the true number of groups')
axes[1].set(xlabel='Number of clusters', ylabel='Total WCSS', xticks=list(ks), title='Elbow curve')
axes[1].legend(frameon=False)
plt.tight_layout()
plt.show()

**Try:** set `SD = 1.5`, then `SD = 2.5`. The groups start to overlap. Is there still a clear elbow, and would you pick k = 3 without knowing the answer?

## 7. Choosing k: the silhouette coefficient

*Slide: Performance: how to choose optimal number of clusters? (silhouette)*

For point $i$: $a_i$ is the average distance to the other points in its own cluster, $b_i$ the average distance to the points of the nearest other cluster, and

$$s_i = \frac{b_i - a_i}{\max(a_i, b_i)} \in [-1, 1].$$

Near 1: well inside its cluster; near 0: on the boundary between two clusters; negative: probably in the wrong cluster. Average $s_i$ over all points and pick the k with the highest average. The code computes $s_i$ for one point by hand first.

In [ ]:
# Silhouette: one point by hand, the average by k, and the value of every point
SD = 0.9     # spread of each blob
K_SHOW = 3   # k for the point-by-point plot on the right
I = 0        # the point computed by hand

X = make_blobs_slides(sd=SD)
labels = KMeans(n_clusters=K_SHOW, n_init=25, random_state=0).fit_predict(X)
d = np.linalg.norm(X - X[I], axis=1)                  # distances from point I to every point
own = labels == labels[I]
a = d[own].sum() / (own.sum() - 1)                    # average over its own cluster, without itself
b = min(d[labels == j].mean() for j in set(labels) - {labels[I]})
s_each = silhouette_samples(X, labels)
print(f'point {I}: a = {a:.2f}, b = {b:.2f}, s = (b - a) / max(a, b) = {(b - a) / max(a, b):.3f}'
      f'   (scikit-learn: {s_each[I]:.3f})')

ks = range(2, 8)
sil_k = [silhouette_score(X, KMeans(n_clusters=k, n_init=25, random_state=0).fit_predict(X)) for k in ks]
best = ks[int(np.argmax(sil_k))]
print(f'highest average silhouette at k = {best}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(ks, sil_k, 'o-', c=NAVY)
axes[0].scatter(best, max(sil_k), s=130, c=GOLD, zorder=3)
axes[0].set(xlabel='Number of clusters', ylabel='Average silhouette', title='Average silhouette by k')
pts = axes[1].scatter(X[:, 0], X[:, 1], c=s_each, cmap='RdBu', vmin=-1, vmax=1, s=16)
fig.colorbar(pts, ax=axes[1], label='silhouette of each point')
axes[1].set_title(f'Silhouette of each point, k = {K_SHOW}')
plt.tight_layout()
plt.show()

**Try:** set `K_SHOW = 5`. K-means now has to cut real groups, and the points along each cut turn pale: their nearest other cluster is the other half of their own group. Then set `SD = 2.5`: the peak is still at k = 3, but barely above its neighbours. How sure would you be of k = 3 without knowing the answer?

## 8. Is the clustering real? Validation

*Slide: Is the clustering real? Validation*

Elbow and silhouette rank values of k; they do not say whether there is any structure at all. Left: k-means on uniformly random points still returns k clusters. Middle: the elbow curve bends on noise too. Right: **stability**. Re-cluster 10 bootstrap samples, label every original point by the bootstrap centroids, and measure the agreement with the full-data clustering by the adjusted Rand index (1 = the same groups, 0 = no better than chance).

In [ ]:
# K-means on noise, the elbow on noise, and stability under resampling
K = 3          # number of clusters for the noise panel and the stability check
N_BOOT = 10    # bootstrap re-clusterings per data set

noise = np.random.default_rng(2027).uniform(0, 10, size=(150, 2))


def stability(X, k, n_boot, seed=2028):
    '''Agreement (adjusted Rand index) between the full-data clustering and n_boot bootstrap re-clusterings.'''
    rng = np.random.default_rng(seed)
    full = KMeans(n_clusters=k, n_init=25, random_state=0).fit_predict(X)
    scores = []
    for _ in range(n_boot):
        boot = X[rng.choice(len(X), len(X), replace=True)]
        km_boot = KMeans(n_clusters=k, n_init=25, random_state=0).fit(boot)
        scores.append(adjusted_rand_score(full, km_boot.predict(X)))
    return np.array(scores)


km = KMeans(n_clusters=K, n_init=25, random_state=0).fit(noise)
wcss_noise = [KMeans(n_clusters=k, n_init=25, random_state=0).fit(noise).inertia_ for k in range(1, 11)]
stab = {'Three blobs': stability(X_blobs, K, N_BOOT), 'Pure noise': stability(noise, K, N_BOOT)}
for name, s in stab.items():
    print(f'{name}: mean agreement {s.mean():.2f} (lowest {s.min():.2f}, highest {s.max():.2f})')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
show(noise, km.labels_, km.cluster_centers_, ax=axes[0], title=f'K-means on pure noise (k = {K})')
axes[1].plot(range(1, 11), wcss_noise, 'o-', c=NAVY)
axes[1].set(xlabel='Number of clusters', ylabel='Total WCSS', xticks=range(1, 11),
            title='... and the elbow curve still bends')
jitter = np.random.default_rng(0)
for i, (name, s) in enumerate(stab.items()):
    axes[2].scatter(i + jitter.uniform(-0.08, 0.08, len(s)), s, c=[BLUE, GOLD][i], s=40)
axes[2].set(xticks=[0, 1], xticklabels=list(stab), xlim=(-0.5, 1.5), ylim=(-0.05, 1.05),
            ylabel='Agreement (adjusted Rand index)', title='Stability under resampling')
plt.tight_layout()
plt.show()

**Try:** set `K = 5`. The blobs are perfectly stable at k = 3, but with five clusters k-means has to cut real groups, and it does not cut them the same way on every sample. Compare the two mean agreements now. A segment that changes with the sample is an artefact.

## 9. Outliers

*Slide: Outliers and clustering*

One extreme point added to the three blobs. Far enough out, it takes a cluster for itself, and k-means merges two real groups to make room. The ringed point is flagged by the IQR rule of the boxplot on the slide: below Q1 − 1.5 × IQR or above Q3 + 1.5 × IQR on any variable.

In [ ]:
# The three blobs with and without one extreme point
OUTLIER = [30, 30]   # position of the extra point
K = 3

X = np.vstack([X_blobs, OUTLIER])
km_without = KMeans(n_clusters=K, n_init=25, random_state=0).fit(X_blobs)
km_with = KMeans(n_clusters=K, n_init=25, random_state=0).fit(X)

q1, q3 = np.percentile(X, [25, 75], axis=0)
iqr = q3 - q1
flagged = ((X < q1 - 1.5 * iqr) | (X > q3 + 1.5 * iqr)).any(axis=1)
print('cluster sizes without the point:', np.bincount(km_without.labels_).tolist())
print('cluster sizes with the point:   ', np.bincount(km_with.labels_).tolist())
print(f'flagged by the IQR rule: points {np.flatnonzero(flagged).tolist()} (the extra point is number {len(X) - 1})')

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
show(X_blobs, km_without.labels_, km_without.cluster_centers_, ax=axes[0], title='Without the outlier')
show(X, km_with.labels_, km_with.cluster_centers_, ax=axes[1], title=f'With one point at {tuple(OUTLIER)}')
axes[1].scatter(X[flagged, 0], X[flagged, 1], s=220, facecolors='none', edgecolors=ROSE, linewidths=1.5)
plt.tight_layout()
plt.show()

**Try:** move the point to `[25, 25]`: it now joins a cluster and pulls that centroid towards it. At `[12, 12]` the IQR rule no longer flags it, because the rule looks at one variable at a time. The slide's rules of thumb: remove an error or a point from another population; keep a natural one, but check how much it moves the clusters.

## 10. Data shapes where K-means fails

*Slide: Data shapes: failures*

Colour = the cluster k-means finds; marker = the true group. Each title gives the agreement with the true groups (adjusted Rand index, 1 = perfect). K-means assumes compact, round clusters of similar size.

In [ ]:
# Three shapes k-means gets wrong
SMALL_N = 15    # points in each small cluster of the first panel (the big one has 100)
OVERLAP = 1.1   # spread of the three blobs in the second panel
STRETCH = 2.2   # length of the ellipses in the third panel (their width is 0.4)

rng1, rng2, rng3 = (np.random.default_rng(s) for s in (432, 433, 434))   # one generator per panel


def ellipse(n, centre, length, width, angle):
    '''n points in an ellipse rotated by angle degrees.'''
    a = np.deg2rad(angle)
    rotation = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])
    return rng3.normal(0, [length, width], size=(n, 2)) @ rotation.T + centre


shapes = {
    'different sizes': [rng1.normal([5, 5], 1.6, (100, 2)), rng1.normal([1.5, 8.5], 0.3, (SMALL_N, 2)),
                        rng1.normal([8.5, 8.5], 0.3, (SMALL_N, 2))],
    'overlapping': [rng2.normal(c, OVERLAP, (60, 2)) for c in ([4, 6], [6, 6], [5, 4.3])],
    'elongated, rotated': [ellipse(60, [2, 7], STRETCH, 0.4, 20), ellipse(60, [5, 5], STRETCH, 0.4, -30),
                           ellipse(60, [8, 7.5], STRETCH, 0.4, 60)],
}
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, (title, groups) in zip(axes, shapes.items()):
    X = np.vstack(groups)
    truth = np.repeat(range(3), [len(g) for g in groups])
    labels = KMeans(n_clusters=3, n_init=25, random_state=0).fit_predict(X)
    for g, marker in enumerate(['s', '^', 'o']):
        ax.scatter(X[truth == g, 0], X[truth == g, 1], c=[COLS[l] for l in labels[truth == g]], marker=marker, s=18)
    ax.set_title(f'{title}\nagreement with the true groups = {agreement(truth, labels):.2f}')
plt.tight_layout()
plt.show()

**Try:** one panel at a time, set `SMALL_N = 100`, `OVERLAP = 0.5` and `STRETCH = 0.4`. Each change removes one problem (unequal sizes, overlap, elongated shape); watch the agreement in that panel's title.

## 11. Special data structures: two moons

*Slide: Special data structures*

Two interleaving half-moons. K-means with k = 2 cuts straight through both. The silhouette still looks respectable, because it also rewards compact, round groups; the agreement with the true moons shows the failure.

In [ ]:
# K-means on two moons
NOISE = 0.08   # how fuzzy the moons are

X_moons, moon = make_moons(n_samples=300, noise=NOISE, random_state=1)
km = KMeans(n_clusters=2, n_init=25, random_state=0).fit(X_moons)
print(f'silhouette = {silhouette_score(X_moons, km.labels_):.3f}, WCSS = {km.inertia_:.1f}, '
      f'agreement with the true moons = {agreement(moon, km.labels_):.2f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
show(X_moons, ax=axes[0], title='Raw data')
show(X_moons, km.labels_, km.cluster_centers_, ax=axes[1], title='K-means (k = 2)')
plt.tight_layout()
plt.show()

**Try:** set `NOISE = 0.02` for crisp moons. Does the agreement improve? The problem is the shape, not the noise.

## 12. DBSCAN

*Slides: DBSCAN*

DBSCAN grows clusters out of dense regions. A point with at least `MIN_SAMPLES` points (itself included) within distance `EPS` is a core point; a cluster is a chain of core points plus their neighbours. Points in no dense region are labelled noise and drawn grey. You do not choose the number of clusters.

In [ ]:
# DBSCAN on the same moons, with eps and minPts as on the slide
EPS = 0.16        # how close must neighbours be?
MIN_SAMPLES = 5   # minPts: points needed for a dense region, the point itself included

X_moons, moon = make_moons(n_samples=300, noise=0.08, random_state=1)
km = KMeans(n_clusters=2, n_init=25, random_state=0).fit(X_moons)
db = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES).fit(X_moons)
print(f'DBSCAN: {len(set(db.labels_) - {-1})} cluster(s) and {(db.labels_ == -1).sum()} noise point(s)')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
show(X_moons, km.labels_, ax=axes[0], title='K-means (k = 2)')
show(X_moons, db.labels_, ax=axes[1], title=f'DBSCAN (eps = {EPS}, minPts = {MIN_SAMPLES})')
plt.tight_layout()
plt.show()

**Try:** set `EPS = 0.08`: the moons break into many small clusters and much of the data becomes noise. Set `EPS = 0.5`: everything merges into one cluster. Choosing eps is the hard part.

DBSCAN is robust to outliers: it labels them noise instead of giving them a cluster. Here it is on the three blobs plus the extreme point of block 9, with an `eps` that suits the blobs' scale. A few stray points at the edges of the blobs become noise too.

In [ ]:
# DBSCAN on the three blobs plus the extreme point of block 9
X = np.vstack([X_blobs, [30, 30]])
db = DBSCAN(eps=1.0, min_samples=5).fit(X)
print(f'{len(set(db.labels_) - {-1})} cluster(s); noise points: {np.flatnonzero(db.labels_ == -1).tolist()}'
      f' (the extra point is number {len(X) - 1})')

plt.figure(figsize=(5.5, 4.5))
show(X, db.labels_, title='DBSCAN (eps = 1.0): the outlier is noise')
plt.show()

## 13. Hierarchical clustering: the dendrogram

*Slides: Hierarchical clustering: agglomerative, and the dendrogram*

Agglomerative clustering starts with every point as its own cluster and repeatedly merges the two closest clusters. "Closest" needs a **linkage** rule: single (nearest pair), complete (farthest pair), average, or Ward (smallest increase in within-cluster variance). The dendrogram records every merge at its height; cutting the tree at a height gives the clusters at that level. The seven points are the slide's.

In [ ]:
# The slide's seven points, a linkage rule, and one cut through the tree
LINKAGE = 'complete'   # 'single', 'complete', 'average' or 'ward'
CUT = 4.0              # height at which the tree is cut

pts = np.array([[1, 1], [3, 3], [1.5, 4], [7, 3], [5, 6], [6, 6], [6.5, 5.5]])
names = [f'p{i}' for i in range(len(pts))]
Z = linkage(pts, method=LINKAGE)                     # Euclidean distance
clusters = fcluster(Z, t=CUT, criterion='distance') - 1
print('merge heights:', Z[:, 2].round(2).tolist())
print('clusters at the cut:', dict(zip(names, clusters.tolist())))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
show(pts, clusters, ax=axes[0], title=f'{len(set(clusters))} cluster(s) at height {CUT}')
for name, (x, y) in zip(names, pts):
    axes[0].annotate(name, (x, y), xytext=(6, 4), textcoords='offset points', color=NAVY)
dendrogram(Z, labels=names, ax=axes[1], link_color_func=lambda k: NAVY)
axes[1].axhline(CUT, ls='--', c=ROSE)
axes[1].set(title=f'Dendrogram ({LINKAGE} linkage)', ylabel='Euclidean distance')
plt.tight_layout()
plt.show()

**Try:** set `CUT = 2.5`, then `CUT = 1`, for more clusters. Back at `CUT = 4.0`, set `LINKAGE = 'single'`: every merge now happens lower, and the same cut leaves a single cluster. The linkage rule, not only the data, shapes the tree.

## 14. Distance and scale decide the clustering

*Slide: Distance and scale decide the clustering*

Simulated firms with two variables in different units: revenue in millions and profit margin in percent. There are two true groups, low-margin (circles) and high-margin (triangles) firms. Unscaled, the distances are almost entirely revenue, so k-means splits on revenue. Standardised (each variable to mean 0 and standard deviation 1), it finds the margin groups.

In [ ]:
# K-means on two financial ratios, unscaled and standardised
REVENUE_UNIT = 1   # 1 = revenue in millions; 0.001 = in billions; 1000 = in thousands

rng = np.random.default_rng(2029)
revenue = rng.lognormal(np.log(400), 0.9, 120) * REVENUE_UNIT
margin = np.r_[rng.normal(4, 1.5, 60), rng.normal(14, 1.5, 60)]
group = np.repeat([0, 1], 60)                         # 0 = low margin, 1 = high margin
firms = np.c_[revenue, margin]

raw = KMeans(n_clusters=2, n_init=25, random_state=0).fit_predict(firms)
scaled = KMeans(n_clusters=2, n_init=25, random_state=0).fit_predict(StandardScaler().fit_transform(firms))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, labels, title in [(axes[0], raw, 'Unscaled'), (axes[1], scaled, 'Standardised')]:
    for g, marker in enumerate(['o', '^']):
        ax.scatter(firms[group == g, 0], firms[group == g, 1], c=[COLS[l] for l in labels[group == g]],
                   marker=marker, s=22)
    ax.set(title=f'{title}: agreement with the true groups = {agreement(group, labels):.2f}',
           xlabel='Revenue', ylabel='Profit margin (%)')
plt.tight_layout()
plt.show()

**Try:** set `REVENUE_UNIT = 0.001`, revenue in billions. Now unscaled k-means finds the margin groups, because the margin has the larger numbers. Nothing about the firms changed, only the unit. The standardised result is the same either way.

## 15. Principal component analysis

*Slides: PCA, and Let's be practical!*

Correlated two-dimensional data. PC1 is the direction along which the data vary most; PC2 is the direction of maximal variance among all directions uncorrelated with PC1 (in two dimensions, the perpendicular one). The arrows start at the mean, and their length is proportional to each component's standard deviation.

In [ ]:
# PCA on correlated two-dimensional data
RHO = 0.75   # correlation between X1 and X2 (variance 4 each, as on the slide)

rng = np.random.default_rng(446)
X = rng.multivariate_normal([5, 5], [[4, 4 * RHO], [4 * RHO, 4]], size=120)
pca = PCA().fit(X)
loadings = np.array([v if v[np.argmax(np.abs(v))] > 0 else -v for v in pca.components_])   # a component's sign is arbitrary
print('share of variance:', pca.explained_variance_ratio_.round(3).tolist())
print('loadings (rows PC1, PC2; columns X1, X2):\n', loadings.round(3))

plt.figure(figsize=(6, 5.5))
plt.scatter(X[:, 0], X[:, 1], s=14, c=BLUE, alpha=0.8)
for name, vec, sd in zip(['PC1', 'PC2'], loadings, np.sqrt(pca.explained_variance_)):
    tip = pca.mean_ + 2.2 * sd * vec
    plt.annotate('', xy=tip, xytext=pca.mean_, arrowprops=dict(arrowstyle='-|>', color=GOLD, lw=2.5))
    plt.text(*tip, f' {name}', color=NAVY, fontweight='bold')
plt.gca().set(xlabel='X1', ylabel='X2', aspect='equal')
plt.show()

**Try:** set `RHO = 0.95`: PC1 carries almost all the variance, so one component would summarise both variables. With `RHO = 0` the two components share the variance about equally, and their directions are arbitrary.

## 16. Scaling matters

*Slide: Scaling matters!*

The US arrests data from the slide: 50 US states in 1973, arrests per 100,000 residents for murder, assault and rape, and the percentage of the population living in cities. Assault has a variance hundreds of times that of murder, so unscaled PCA is essentially Assault. Standardised, every variable enters with variance 1 and all four contribute.

In [ ]:
# PCA biplots of the US arrests data, standardised and unscaled
DROP = []   # variables to leave out, e.g. ['Assault']

arrests = pd.read_csv(RAW + 'data/USArrests.csv', index_col='State').drop(columns=DROP)
print('variances:', arrests.var().round(1).to_dict())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, scale in zip(axes, [True, False]):
    data = StandardScaler().fit_transform(arrests) if scale else arrests.to_numpy()
    pca = PCA(n_components=2).fit(data)
    scores = pca.transform(data)
    ax.scatter(scores[:, 0], scores[:, 1], s=16, c=BLUE, alpha=0.8)
    stretch = 0.8 * np.abs(scores).max(axis=0) / np.abs(pca.components_).max(axis=1)   # loadings drawn at the scores' range, per axis
    tips = pca.components_.T * stretch
    ax.scatter(tips[:, 0], tips[:, 1], s=0)   # invisible, so that the axes include every arrow tip
    ax.margins(0.12)
    for var, (x, y) in zip(arrests.columns, tips):
        ax.annotate('', xy=(x, y), xytext=(0, 0), arrowprops=dict(arrowstyle='-|>', color='#B8860B', lw=2))
        ax.text(x, y, f' {var}', color='#8A6508', fontweight='bold')
    share = pca.explained_variance_ratio_
    ax.set(title=f"{'Standardised' if scale else 'Unscaled'}: PC1 {share[0]:.0%}, PC2 {share[1]:.0%} of the variance",
           xlabel='First principal component', ylabel='Second principal component')
plt.tight_layout()
plt.show()

**Try:** set `DROP = ['Assault']` and look at the unscaled plot again. Which variable takes over now, and why?

## What to take away

- K-means is two steps repeated: assign each point to its nearest centroid, move each centroid to the mean of its points.
- Its result depends on k, on the start (use many starts), on the units (scale first) and on the shape of the groups.
- Elbow and silhouette help choose k; they do not show that clusters exist. Check stability, and look at the data.
- DBSCAN (density) and hierarchical clustering (a tree of merges) make other assumptions. PCA reduces the number of variables, and it too depends on scaling.

For more, `04_EAIF_Comparison_clustering_methods.ipynb` compares four clustering methods on six toy data sets, and `04_EAIF_Clustering_in_Finance.ipynb` applies clustering to credit risk and company valuation.